In [1]:
import warnings
from rdkit import RDLogger

# 屏蔽 RDKit 警告
RDLogger.DisableLog('rdApp.*')

# 或屏蔽所有 Python 警告
warnings.filterwarnings("ignore")
# 屏蔽 LightGBM 警告
warnings.filterwarnings("ignore", category=UserWarning, module="lightgbm")

In [2]:
import torch
from sklearn.model_selection import StratifiedKFold
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics import precision_recall_curve, auc
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import joblib
import optuna
from rdkit.Chem import Descriptors, AllChem
from tqdm import tqdm  # 导入tqdm
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold





In [3]:
# 函数：将SMILES转换为分子描述符和指纹
def smiles_to_features(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    # 提取描述符
    descriptors = [
        Descriptors.MolWt(mol),  # 分子量
        Descriptors.MolLogP(mol),  # LogP
        Descriptors.NumHDonors(mol),  # 氢键供体数量
        Descriptors.NumHAcceptors(mol)  # 氢键受体数量
    ]
    # 生成Morgan指纹
    fingerprint = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
    fingerprint_array = np.zeros((2048,))
    Chem.DataStructs.ConvertToNumpyArray(fingerprint, fingerprint_array)
    # 合并描述符和指纹
    features = np.concatenate([descriptors, fingerprint_array])
    return features


In [12]:

from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold
from tqdm import tqdm
import optuna
import numpy as np

def train_evaluate_regression_model_with_optuna(model_name, model_class, param_func, X, y, groups):
    def objective(trial):
        params = param_func(trial)
        model = model_class(**params)

        gkf = GroupKFold(n_splits=10)
        maes = []

        for train_idx, val_idx in tqdm(gkf.split(X, y, groups=groups), total=10, desc=f"Training {model_name}"):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)

            # ✅ 计算 MAE
            mae = mean_absolute_error(y_val, y_pred)
            maes.append(mae)

        return np.mean(maes)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=30)

    print(f'Best parameters for {model_name}: {study.best_params}')
    print(f'Best mean MAE: {study.best_value:.4f}')


    

In [4]:
# 数据预处理
df = pd.read_excel('../invertebrates_LC50_unique.xlsx')
labels = df['mgperL'].values
smiles_list = df['SMILES_Canonical_RDKit'].tolist()
endpoints_a = df['endpoint']
Duration_Values_a = df['Duration_Value'].values
effects_a = df['effect']


In [5]:

features = []
new_labels = []
new_smiles_list = []
endpoints = []
Duration_Values = []
effects =[]


for smiles, label,a,b,c in zip(smiles_list, labels,Duration_Values_a,effects_a,endpoints_a):
    feature = smiles_to_features(smiles)
    if feature is not None:
        features.append(feature)
        new_labels.append(label)
        new_smiles_list.append(smiles)
        Duration_Values.append(a)
        effects.append(b)
        endpoints.append(c)

X = np.array(features)
y = np.array(new_labels)
groups = new_smiles_list  # 可直接用于 GroupKFold




In [6]:
def encode_column(zz):
    zz_series = pd.Series(zz)  # 转换为 Series
    unique_values = zz_series.unique()
    if len(unique_values) > 1:
        encoder = OneHotEncoder(sparse_output=False)
        return encoder.fit_transform(zz_series.values.reshape(-1, 1))
    else:
        return None  # 只有一种类别时忽略

Duration_Values =pd.Series(Duration_Values)


# 编码 effect、endpoint 和 species_group 列
effect_encoded = encode_column(effects)
endpoint_encoded = encode_column(endpoints)
#species_encoded = encode_column(df, 'species_group')

# # 将需要的列拼接成输入 X
X = np.hstack((X, Duration_Values.values.reshape(-1, 1)))

# # 拼接编码后的列（如果存在）
for encoded_feature in [effect_encoded, endpoint_encoded]:
     if encoded_feature is not None:
         X = np.hstack((X, encoded_feature))



y=np.log1p(y)

In [13]:
def xgb_param_func(trial):
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),   # L1 正则
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0)  # L2 正则
    }
from xgboost import XGBRegressor

train_evaluate_regression_model_with_optuna(
    "XGBoost",
    XGBRegressor,
    xgb_param_func,
    X, y, groups
)

[I 2025-05-15 19:01:07,097] A new study created in memory with name: no-name-d5e9cb4e-04ae-42bc-ad50-e3d54afd85c5
Training XGBoost: 100%|██████████| 10/10 [01:00<00:00,  6.01s/it]
[I 2025-05-15 19:02:07,250] Trial 0 finished with value: 1.0018584224848242 and parameters: {'n_estimators': 539, 'max_depth': 11, 'learning_rate': 0.038450928940930024, 'subsample': 0.6237724674838361, 'colsample_bytree': 0.7300982641057816, 'reg_alpha': 0.126766432101028, 'reg_lambda': 0.14839204699131747}. Best is trial 0 with value: 1.0018584224848242.
Training XGBoost: 100%|██████████| 10/10 [02:03<00:00, 12.35s/it]
[I 2025-05-15 19:04:10,718] Trial 1 finished with value: 1.0112484724897548 and parameters: {'n_estimators': 529, 'max_depth': 20, 'learning_rate': 0.07982549390399164, 'subsample': 0.6382604423310759, 'colsample_bytree': 0.9967230233582167, 'reg_alpha': 0.18824486212970315, 'reg_lambda': 0.6563677646504009}. Best is trial 0 with value: 1.0018584224848242.
Training XGBoost: 100%|██████████| 1

Best parameters for XGBoost: {'n_estimators': 386, 'max_depth': 18, 'learning_rate': 0.028539528553053656, 'subsample': 0.7507155150876466, 'colsample_bytree': 0.9258878058674596, 'reg_alpha': 0.4460214564022941, 'reg_lambda': 0.9209669677939677}
Best mean MAE: 0.9941


In [14]:
from lightgbm import LGBMRegressor

def lgbm_param_func(trial):
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        'verbose': -1
    }

print("Training LightGBM (Poisson)...")
train_evaluate_regression_model_with_optuna(
    "LightGBM",
    lambda **params: LGBMRegressor(objective="poisson", **params),  # ✅ 加入 Poisson 目标
    lgbm_param_func,
    X, y, groups
)

[I 2025-05-15 19:33:19,632] A new study created in memory with name: no-name-92689051-cf65-4ed8-b412-0dd9a8ead073


Training LightGBM (Poisson)...


Training LightGBM: 100%|██████████| 10/10 [00:15<00:00,  1.54s/it]
[I 2025-05-15 19:33:35,036] Trial 0 finished with value: 1.0235689051449204 and parameters: {'n_estimators': 485, 'max_depth': 14, 'num_leaves': 272, 'learning_rate': 0.2462863629493069, 'feature_fraction': 0.9101669059164736, 'bagging_fraction': 0.691233090494175, 'bagging_freq': 1, 'reg_alpha': 0.1705954217798159, 'reg_lambda': 0.2660834565265957}. Best is trial 0 with value: 1.0235689051449204.
Training LightGBM: 100%|██████████| 10/10 [00:11<00:00,  1.14s/it]
[I 2025-05-15 19:33:46,414] Trial 1 finished with value: 1.0027128107487258 and parameters: {'n_estimators': 373, 'max_depth': 14, 'num_leaves': 75, 'learning_rate': 0.2590029939566883, 'feature_fraction': 0.9317332250194612, 'bagging_fraction': 0.843320600497117, 'bagging_freq': 7, 'reg_alpha': 0.02699503149392679, 'reg_lambda': 0.7426232517547302}. Best is trial 1 with value: 1.0027128107487258.
Training LightGBM: 100%|██████████| 10/10 [00:16<00:00,  1.63s/i

Best parameters for LightGBM: {'n_estimators': 357, 'max_depth': 17, 'num_leaves': 80, 'learning_rate': 0.09917864110541515, 'feature_fraction': 0.8015728402355902, 'bagging_fraction': 0.9220720677020435, 'bagging_freq': 4, 'reg_alpha': 0.19557544532177482, 'reg_lambda': 0.29784078625110505}
Best mean MAE: 0.9798


In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
import optuna
import numpy as np


class DNNWithSoftplus(nn.Module):
    def __init__(self, input_dim, hidden_sizes, activation):
        super().__init__()
        act_fn = {
            'relu': nn.ReLU(),
            'logistic': nn.Sigmoid(),
            'tanh': nn.Tanh()
        }[activation]
        layers = []
        prev_dim = input_dim
        for h in hidden_sizes:
            layers += [nn.Linear(prev_dim, h), act_fn]
            prev_dim = h
        layers += [nn.Linear(prev_dim, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return F.softplus(self.net(x)).squeeze(-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def train_dnn_with_optuna_pytorch(X, y, groups, device=device):
    def dnn_param_func(trial):
        return {
            'hidden_layer_sizes': trial.suggest_categorical(
                'hidden_layer_sizes', [(50,), (100,), (150,), (100, 50), (150, 100, 50)]
            ),
            'activation': trial.suggest_categorical('activation', ['relu', 'logistic', 'tanh']),
            'alpha': trial.suggest_float('alpha', 1e-5, 1e-2, log=True),
            'learning_rate': trial.suggest_float('learning_rate_init', 1e-4, 1e-2, log=True),
            'optimizer': trial.suggest_categorical('solver', ['adam', 'sgd'])
        }

    def objective(trial):
        params = dnn_param_func(trial)
        model = DNNWithSoftplus(
            input_dim=X.shape[1],
            hidden_sizes=params['hidden_layer_sizes'],
            activation=params['activation']
        ).to(device)

        optimizer = {
            'adam': torch.optim.Adam,
            'sgd': torch.optim.SGD
        }[params['optimizer']](model.parameters(), lr=params['learning_rate'], weight_decay=params['alpha'])

        loss_fn = nn.MSELoss()
        gkf = GroupKFold(n_splits=10)
        fold_maes = []

        for train_idx, val_idx in gkf.split(X, y, groups=groups):
            X_train, y_train = X[train_idx], y[train_idx]
            X_val, y_val = X[val_idx], y[val_idx]

            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_val = scaler.transform(X_val)

            train_ds = TensorDataset(torch.tensor(X_train).float(), torch.tensor(y_train).float())
            train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)

            model.train()
            for epoch in range(100):
                for xb, yb in train_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    optimizer.zero_grad()
                    pred = model(xb)
                    loss = loss_fn(pred, yb)
                    loss.backward()
                    optimizer.step()

            model.eval()
            with torch.no_grad():
                val_preds = model(torch.tensor(X_val).float().to(device)).cpu().numpy()
                mae = mean_absolute_error(y_val, val_preds)
                fold_maes.append(mae)

        return np.mean(fold_maes)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=30)
    print("\n✅ Best Parameters Found:")
    print(study.best_params)
    print(f"Mean MAE = {study.best_value:.4f}")
    return study.best_params


best_dnn_params = train_dnn_with_optuna_pytorch(X, y, groups)

[I 2025-05-15 16:17:32,000] A new study created in memory with name: no-name-5ba1abca-bc1e-407a-b411-bd1f43570543
[I 2025-05-15 16:18:18,606] Trial 0 finished with value: 0.8947501837020082 and parameters: {'hidden_layer_sizes': (100,), 'activation': 'logistic', 'alpha': 0.0002167712451453112, 'learning_rate_init': 0.0007199787415960483, 'solver': 'sgd'}. Best is trial 0 with value: 0.8947501837020082.
[I 2025-05-15 16:19:10,000] Trial 1 finished with value: 0.6153813975203148 and parameters: {'hidden_layer_sizes': (50,), 'activation': 'logistic', 'alpha': 1.6146191890321593e-05, 'learning_rate_init': 0.0002451271462417534, 'solver': 'adam'}. Best is trial 1 with value: 0.6153813975203148.
[I 2025-05-15 16:20:05,668] Trial 2 finished with value: 0.6511811707855875 and parameters: {'hidden_layer_sizes': (150, 100, 50), 'activation': 'logistic', 'alpha': 1.070907372378741e-05, 'learning_rate_init': 0.0001929412441925032, 'solver': 'adam'}. Best is trial 1 with value: 0.6153813975203148.



✅ Best Parameters Found:
{'hidden_layer_sizes': (100, 50), 'activation': 'relu', 'alpha': 0.0003442047658029931, 'learning_rate_init': 0.00010253643511316506, 'solver': 'adam'}
Mean MAE = 0.5385
